# 07m — Trackintel semantic-only parity

Independent open-source comparator for the frozen CP2-v2 HOME/OFFICE model.

Primary design:

```text
frozen 5,821 CP1 stays
        ↓
CP2-v2 per-stay local timezone
        ↓
same complete-link 200 m location_id
        ↓
Trackintel 1.4.2
  FREQ / OSNA
  pre_filter=False
        ↓
candidate-identity comparison
```

This stage does **not** replace Trackintel staypoint/location generation and does **not** retune production. Keeping the exact production stay and location namespace isolates differences in the semantic rule itself.

Trackintel documentation: FREQ assigns the most visited location to home and the second to work; OSNA uses weekday rest/work/leisure timeframes. The comparator disables Trackintel's native pre-filter intentionally so its eligibility thresholds do not confound semantic-rule parity.

In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import os
import subprocess
import sys

import numpy as np
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
TRACKINTEL_VERSION='1.4.2'

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

try:
    installed_ti=importlib.metadata.version('trackintel')
except importlib.metadata.PackageNotFoundError:
    installed_ti=None
if installed_ti != TRACKINTEL_VERSION:
    print(f'Installing trackintel=={TRACKINTEL_VERSION} (current={installed_ti}) ...')
    run([sys.executable,'-m','pip','install','-q','--upgrade',f'trackintel=={TRACKINTEL_VERSION}'])

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's07m',
    REPO_DIR/'analysis'/'07m_trackintel_semantic_parity.py',
)
assert spec is not None and spec.loader is not None
s07m=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07m
spec.loader.exec_module(s07m)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print('trackintel:',importlib.metadata.version('trackintel'))
print(s07m.synthetic_self_check())

## 1. Frozen CP1 + CP2-v2 parity gate

The comparator must start from the same private 5,821-stay CP1 cache and reproduce the frozen CP2-v2 namespace before Trackintel is allowed to run.

Expected gates:

- CP1 stays: 5,821 / 136 users;
- semantic locations: 2,015;
- recurring locations: 716 / 104 users;
- production HOME: 27;
- production OFFICE: 16.

In [ ]:
V=Path('/mnt/geolife-data')
C=V/'cache'/'cp2_v2'/'07m_trackintel_semantic_parity'
C.mkdir(parents=True,exist_ok=True)

def find_cache(name):
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/name,
        V/'artifacts'/'03a'/name,
        REPO_DIR/'artifacts'/'03a'/name,
        V/'cache'/'cp2_home_office'/name,
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob(f'**/{name}'))
    if hits:
        return hits[0]
    raise FileNotFoundError(name)

STAYS_PATH=find_cache('stays_baseline_v1.pkl')
stays=pd.read_pickle(STAYS_PATH)
assert len(stays)==5821
assert stays['user_id'].astype(str).nunique()==136

cfg=HomeOfficeConfig()
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)

label_counts=production['label'].value_counts().to_dict()
recurring=locations.loc[locations['stay_count'].ge(2)]

assert len(semantic_stays)==5821
assert semantic_stays['user_id'].astype(str).nunique()==136
assert len(locations)==2015
assert len(recurring)==716
assert recurring['user_id'].astype(str).nunique()==104
assert label_counts.get('HOME',0)==27
assert label_counts.get('OFFICE',0)==16

parity=pd.DataFrame([{
    'cp1_stays':len(stays),
    'stay_users':stays['user_id'].astype(str).nunique(),
    'semantic_stays':len(semantic_stays),
    'semantic_locations':len(locations),
    'recurring_locations':len(recurring),
    'recurring_location_users':recurring['user_id'].astype(str).nunique(),
    'HOME':label_counts.get('HOME',0),
    'OFFICE':label_counts.get('OFFICE',0),
}])
print('frozen CP2-v2 parity: PASS')
print('stays:',STAYS_PATH)
display(parity)

## 2. Local-wall-clock adapter for Trackintel

CP2-v2 permits a different IANA timezone for every stay. Trackintel expects timezone-aware pandas datetimes in a common datetime dtype.

For this semantic-only comparator, each stay's **local wall-clock start** is encoded as a dummy UTC timestamp, while the true elapsed `duration_s` is preserved. Therefore Trackintel sees the same local weekday/hour behavior without reverting the project to blanket UTC+8.

`finish_wall_delta_s` audits the rare case where the exact local departure wall clock differs from start + elapsed duration, e.g. a DST transition. The dummy timestamps below are semantic-clock carriers only and must not be interpreted as physical UTC timestamps.

In [ ]:
import geopandas as gpd
import trackintel as ti

adapter=s07m.build_trackintel_adapter(semantic_stays)
adapter_diag=s07m.adapter_diagnostics(adapter)
assert len(adapter)==5821
assert adapter['user_id'].nunique()==136
assert adapter[['user_id','location_id']].drop_duplicates().shape[0]==2015

gdf=gpd.GeoDataFrame(
    adapter.copy(),
    geometry=gpd.points_from_xy(adapter['longitude'],adapter['latitude']),
    crs='EPSG:4326',
).rename_geometry('geom')
sp=ti.Staypoints(gdf)
sp.index.name='staypoint_id'

assert len(sp)==5821
assert 'location_id' in sp.columns
print('Trackintel staypoint adapter: PASS')
display(adapter_diag)

## 3. Trackintel FREQ / OSNA — same location namespace

`pre_filter=False` is the primary experiment.

Why: Trackintel's native pre-filter introduces additional thresholds for user/location/staypoint support. Those are valid library defaults, but enabling them here would mix semantic-rule differences with cohort-eligibility differences.

In [ ]:
def fresh_trackintel_staypoints():
    return ti.Staypoints(gdf.copy())

candidate_frames=[]
for method in ['FREQ','OSNA']:
    labeled=ti.analysis.location_identifier(
        fresh_trackintel_staypoints(),
        method=method,
        pre_filter=False,
    )
    candidates=s07m.extract_trackintel_candidates(labeled,method)
    candidate_frames.append(candidates)
    print(method,'candidate rows:',len(candidates),
          'HOME:',int(candidates['label'].eq('HOME').sum()),
          'WORK:',int(candidates['label'].eq('OFFICE').sum()))

trackintel_candidates=pd.concat(candidate_frames,ignore_index=True)
assert set(trackintel_candidates['method'])=={'FREQ','OSNA'}
print('Trackintel semantic methods: PASS')

## 4. Candidate identity parity

These are agreement/coverage diagnostics, **not accuracy**. GeoLife has no HOME/OFFICE ground truth.

The most important distinction is:

- `both_same`: both systems emit/select exactly the same production location;
- `both_different`: both select a candidate, but location identity differs;
- `production_only`: production emits while the comparator does not;
- `comparator_only`: Trackintel selects while production abstains;
- `neither`: neither selects/emits.

In [ ]:
comparison=s07m.build_candidate_comparison(
    semantic_stays['user_id'].astype(str).unique(),
    production,
    trackintel_candidates,
)
agreement=s07m.summarize_agreement(comparison)
status_distribution=s07m.summarize_status_distribution(comparison)
office_reference=s07m.summarize_reference_label(comparison,'OFFICE')
home_reference=s07m.summarize_reference_label(comparison,'HOME')

print('AGREEMENT SUMMARY')
display(agreement)

print('STATUS DISTRIBUTION')
display(status_distribution)

print('FROZEN OFFICE-16 REFERENCE')
display(office_reference)

print('FROZEN HOME-27 REFERENCE')
display(home_reference)

## 5. Stage-07j near-miss diagnostic (optional but expected on the current Volume)

If the Stage-07j private panel exists, check whether Trackintel selects the **same candidate location** as WORK for the nine one-step OFFICE near-misses.

This does not reopen Stage 07l. Even a high Trackintel match rate means only that a simpler external heuristic would choose those candidates; it is not permission to relax the frozen OFFICE gate.

In [ ]:
NEAR_PATH=V/'cache'/'cp2_v2'/'07j_near_miss_office_audit'/'near_miss_office_audit_private.pkl'

near_private=pd.DataFrame()
near_summary=pd.DataFrame()
if NEAR_PATH.exists():
    near_panel=pd.read_pickle(NEAR_PATH)
    assert len(near_panel)==25
    assert int(near_panel['audit_group'].ne('baseline').sum())==9
    near_private,near_summary=s07m.summarize_near_miss_work(
        near_panel,
        trackintel_candidates,
    )
    print('TRACKINTEL WORK × STAGE-07j NEAR-MISS')
    display(near_summary)
else:
    print('Stage-07j private panel not found; near-miss diagnostic skipped:',NEAR_PATH)

## 6. Trackintel FREQ × OSNA internal agreement

This view is independent of production and shows whether the two Trackintel semantic heuristics themselves select the same location identity.

In [ ]:
pivot=trackintel_candidates.pivot_table(
    index=['user_id','label'],
    columns='method',
    values='location_id',
    aggfunc='first',
)
for method in ['FREQ','OSNA']:
    if method not in pivot.columns:
        pivot[method]=pd.NA
pivot=pivot.reset_index()
pivot['both_selected']=pivot['FREQ'].notna() & pivot['OSNA'].notna()
pivot['same_location']=pivot['both_selected'] & pivot['FREQ'].eq(pivot['OSNA'])

cross_method=(
    pivot.groupby('label',as_index=False)
    .agg(
        user_label_rows=('user_id','size'),
        freq_selected=('FREQ',lambda s:int(s.notna().sum())),
        osna_selected=('OSNA',lambda s:int(s.notna().sum())),
        both_selected=('both_selected','sum'),
        same_location=('same_location','sum'),
    )
)
cross_method['same_rate_among_both']=np.where(
    cross_method['both_selected'].gt(0),
    cross_method['same_location']/cross_method['both_selected'],
    np.nan,
)
display(cross_method)

## 7. Save outputs to Modal Volume

User-level candidate identity remains private on the Volume. Aggregate tables are safe for project documentation.

In [ ]:
# Private
trackintel_candidates.to_pickle(C/'trackintel_candidates_private.pkl')
comparison.to_pickle(C/'production_trackintel_comparison_private.pkl')
pivot.to_pickle(C/'trackintel_cross_method_private.pkl')
if len(near_private):
    near_private.to_pickle(C/'near_miss_trackintel_work_private.pkl')

# Aggregate
parity.to_csv(C/'production_parity.csv',index=False)
adapter_diag.to_csv(C/'adapter_diagnostics.csv',index=False)
agreement.to_csv(C/'agreement_summary.csv',index=False)
status_distribution.to_csv(C/'status_distribution.csv',index=False)
office_reference.to_csv(C/'office_reference_summary.csv',index=False)
home_reference.to_csv(C/'home_reference_summary.csv',index=False)
cross_method.to_csv(C/'trackintel_cross_method_summary.csv',index=False)
if len(near_summary):
    near_summary.to_csv(C/'near_miss_work_summary.csv',index=False)

print('saved:',C)

## Interpretation boundary

- Trackintel 1.4.2 is an **independent implementation comparator**, not semantic ground truth.
- FREQ/OSNA run on the exact production CP1 stays and exact CP2-v2 `location_id` namespace.
- `pre_filter=False` is deliberate so this stage isolates semantic selection rather than Trackintel eligibility gates.
- Dummy UTC timestamps preserve per-stay local wall-clock semantics; they are not real UTC times.
- Agreement is evidence of methodological convergence, not correctness.
- Disagreement is a failure-mode diagnostic, not evidence that either implementation is wrong.
- Trackintel selection of a Stage-07j near-miss does not override split-half, held-out, recurrence, BCL, or Stage-07l imagery evidence.
- No HOME/OFFICE production threshold changes are allowed from Stage 07m alone.